In [1]:
import pandas as pd
import re
import emoji
import json
from tqdm import tqdm

In [2]:
df = pd.read_csv("../datasets/tweets.csv")
df.drop(df[df['language'] != 'en'].index, inplace=True)
df = df.drop_duplicates(subset='content', keep='first')
df = df.iloc[125440:].reset_index(drop=True)

C:\Users\deepp\AppData\Local\Temp\ipykernel_26932\232719046.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../datasets/tweets.csv")


### Pre-processing

In [3]:
def clean_tweet(tweet):
    tweet = re.sub(r"http\S+|www\S+|https\S+", " ", tweet, flags=re.MULTILINE)
    tweet = re.sub(r"@|#|:|_", " ", tweet)
    tweet = re.sub(r",", ", ", tweet)
    tweet = re.sub(r"!", "! ", tweet)
    tweet = re.sub(r"\?", "? ", tweet)
    tweet = re.sub(r";", "; ", tweet)
    tweet = re.sub(r"[^\w\s.,!?;]", " ", tweet)
    tweet = emoji.replace_emoji(tweet, " ")  
    tweet = re.sub(r"(?<!\d)\.|(?<=\d)\.(?!\d)|(?<!\d)\.(?!\d)", ". ", tweet)
    tweet = re.sub(r"(?<=\d)\. (?=\d)", ".", tweet)
    tweet = re.sub(r"\s+", " ", tweet).strip()
    return tweet

def is_ascii(text):
    return all(ord(char) < 128 for char in text)

df['content'] = df['content'].apply(clean_tweet)
df = df[df['content'].apply(is_ascii)]

df = df.reset_index(drop=True)

df


,date,content,hashtags,like_count,rt_count,followers_count,isVerified,language,coordinates,place,source
0,2023-02-06 22:17:28+00:00,"You have 10 boyfriends , Yet when you pray you...","['earthquake', 'BBTitians', 'PrayForTurkey', '...",0.0,0.0,436.0,False,en,NaN,NaN,Twitter for Android
1,2023-02-06 22:17:27+00:00,China is willing to provide emergency humanita...,"['damascus', 'istanbul', 'turkey', 'ankara', '...",0.0,0.0,531.0,False,en,NaN,NaN,IdeallyaNews
2,2023-02-06 22:17:27+00:00,"Earthquake kills more than 1, 300 in southern ...","['more_than', 'istanbul', 'turkey', 'ankara', ...",0.0,0.0,531.0,False,en,NaN,NaN,IdeallyaNews
3,2023-02-06 22:17:25+00:00,THOUSANDS of people in Syria and Turkey are un...,"['Syria', 'Turkey']",0.0,0.0,0.0,False,en,NaN,NaN,Twitter for iPhone
4,2023-02-06 22:17:24+00:00,Heart heavy for Turkey Syria Lebanon and all t...,"['Turkey', 'Syria', 'Lebanon', 'earthquake', '...",1.0,1.0,47.0,False,en,NaN,NaN,Twitter for iPhone
...,...,...,...,...,...,...,...,...,...,...,...
53811,2023-02-06 00:04:02+00:00,All they toys experimented in Syria for years ...,"['Syria', 'Ukraine', 'February2023']",0.0,0.0,679.0,False,en,NaN,NaN,Twitter for iPhone
53812,2023-02-06 00:04:00+00:00,Author Birol Bahadir is very busy these days g...,"['immigrant', 'germany', 'turkey', 'internatio...",0.0,0.0,1362.0,False,en,NaN,NaN,eClincher
53813,2023-02-06 00:01:25+00:00,Earthquake sismo M2.8 strikes 72 km NE of Cala...,"['Earthquake', 'sismo', 'Calama', 'Chile']",3.0,2.0,44544.0,False,en,NaN,NaN,emsc-csem
53814,2023-02-06 00:01:03+00:00,From Istanbul to New York Discovering The Last...,"['ottomanempire', 'ottoman', 'osman', 'osmanbe...",0.0,0.0,66.0,False,en,NaN,NaN,Twitter for Android


In [4]:
def find_non_ascii_chars(tweet):
    return set(char for char in tweet if ord(char) >= 128)

non_ascii_chars = set()
for tweet in df['content']:
    non_ascii_chars.update(find_non_ascii_chars(tweet))

print("Unique non-ASCII characters found in the tweets:")
print(non_ascii_chars)


Unique non-ASCII characters found in the tweets:
set()


In [5]:
def process_hashtags(hashtags):
    if isinstance(hashtags, str): 
        return [tag.strip().strip(']').strip('[').strip("'") for tag in hashtags.split(",")]
    return [] 

In [6]:
jp = pd.read_csv('../datasets/jp.csv')
city = pd.read_csv('../datasets/city.csv')
valid_locations = pd.read_csv('verified_valid_gpe.csv')
countries = pd.read_csv('../datasets/countries.csv')

C:\Users\deepp\AppData\Local\Temp\ipykernel_26932\3131224516.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  jp = pd.read_csv('../datasets/jp.csv')


In [7]:
a = list(city['name']) + list(countries['Country'])
city_list = list(set(a))

In [8]:
len(city_list)

462

In [9]:
jsonl_data = []

for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
    content = row["content"]
    hashtags = process_hashtags(row["hashtags"]) + city_list
    hashtags = list(set(hashtags))
    # hashtags = [item for item in hashtags if item.lower() in valid_locations['valid_gpe'].str.lower().values]

    entities = []
    
    for hashtag in hashtags:
        if isinstance(hashtag, str):
            lowercase_hashtag = hashtag.lower()
        else:
            lowercase_hashtag = str(hashtag).lower()
        
        pattern = r'\b' + re.escape(lowercase_hashtag) + r'\b'
        matches = re.finditer(pattern, content.lower())
        for match in matches:
            start_idx = match.start()
            end_idx = match.end()
            if [start_idx, end_idx, "GPE"] not in entities:
                entities.append([start_idx, end_idx, "GPE"])
    
    jsonl_data.append([f"{content} ", {"entities": entities}])

with open("../datasets/train/1xtagged_gpe.jsonl", "w", encoding="utf-8") as jsonl_file:
    for entry in jsonl_data:
        json.dump(entry, jsonl_file)
        jsonl_file.write("\n")

Processing rows: 100%|██████████| 53816/53816 [03:00<00:00, 298.21it/s]


In [10]:
b = city_list + list(valid_locations['valid_gpe'])

In [11]:
b

['Sūrat',
 'Comilla',
 'Assam',
 'Western Visayas',
 'Republic of Uzbekistan',
 'New York',
 'England',
 'Uttar Pradesh',
 'Indiana',
 'Orientale Province',
 'Jiangxi Sheng',
 'Ōsaka-fu',
 'Daerah Khusus Ibukota Jakarta',
 'Grand-Est',
 'Southern Region',
 'Zimbabwe',
 'Republic of Kenya',
 'Democratic Socialist Republic of Sri Lanka',
 'Ceará',
 'Guizhou Sheng',
 'Republic of Turkey',
 'Nashik Division',
 'Dhaka',
 'Rhône-Alpes',
 'Gansu Sheng',
 'National Capital Region',
 'Ethiopia',
 'Western Province',
 'Banten',
 'Sweden',
 'Hokkaido',
 'Seoul',
 'Federal Republic of Germany',
 'Republic of Benin',
 'Virginia',
 'Cambodia',
 'Murshidabad',
 'Comunitat Valenciana',
 'Republic of Kazakhstan',
 'Shanxi Sheng',
 'Chongqing Shi',
 'Republic of Tajikistan',
 'Ciudad de México',
 'Malawi',
 'People’s Democratic Republic of Algeria',
 'Republic of Ecuador',
 'Republic of Mali',
 'Kingdom of Morocco',
 'Liaoning Sheng',
 'Hunan Sheng',
 'Santa Catarina',
 'Ganzhou Shi',
 'Provinsi Jawa Te

In [12]:
gpe_set = set(b)

In [13]:
def remove_invalid_gpes(input_file, output_file, valid_gpes):
    valid_gpes_lower = {gpe.lower() for gpe in valid_gpes}

    with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
        for line in infile:
            tweet = json.loads(line)
            
            text = tweet[0]  
            entities = tweet[1]["entities"]  

            filtered_entities = [
                (start, end, label) for start, end, label in entities
                if not (label == "GPE" and text[start:end].strip().lower() not in valid_gpes_lower)
            ]
            
            result = [text, {"entities": filtered_entities}]
            json.dump(result, outfile, ensure_ascii=False)
            outfile.write("\n")

In [14]:
input_file = "../datasets/train/1xtagged_gpe.jsonl"  
output_file = "../datasets/train/1xfiltered_tagged_gpe.jsonl"  

remove_invalid_gpes(input_file, output_file, gpe_set)

In [15]:
disaster_keywords = [
    "earthquake", "tremor", "aftershock", "seismic", "fault", "epicenter",
    "magnitude", "Richter scale", "shaking", "ground", "quake", "foreshock",
    "tectonic", "plate", "shockwave", "aftermath", "felt", "feel",
    "strong", "massive", "devastating", "violent", "powerful", "intense",
    "mild", "deep", "surface", "shallow", "damage", "collapse", "ruins", "wreckage",
    "destroyed", "cracks", "crumbling", "impact", "disaster", "displaced",
    "homeless", "injury", "injuries", "fatalities", "debris", "rubble", "casualties",
    "trapped", "death", "died", "alert", "warning", "evacuation", "rescue",
    "search", "emergency", "relief", "assistance", "volunteers", "preparedness",
    "shelter", "efforts", "response team", "seismograph", "seismology",
    "intensity", "measurement", "USGS", "depth", "geological", "seismometer",
    "tsunami", "landslide", "fire", "eruption", "volcano", "flood", "pray", "thoughts",
    "fear", "panic", "trauma", "loss", "tragedy", "devastation", "solidarity", "support"
]

In [16]:
input_file = "../datasets/train/1xfiltered_tagged_gpe.jsonl"  
jsonl_data = []

with open(input_file, "r", encoding="utf-8") as file:
    for line in file:
        jsonl_data.append(json.loads(line))

updated_jsonl_data = []

for entry in tqdm(jsonl_data, desc="Processing rows"):
    content = entry[0]
    entities = entry[1]["entities"]

    for keyword in disaster_keywords:
        if isinstance(keyword, str):
            lowercase_keyword = keyword.lower()
        else:
            lowercase_keyword = str(keyword).lower()
        
        pattern = r'\b' + re.escape(lowercase_keyword) + r'\b'
        matches = re.finditer(pattern, content.lower())
        for match in matches:
            start_idx = match.start()
            end_idx = match.end()
            if [start_idx, end_idx, "DISASTER"] not in entities:
                entities.append([start_idx, end_idx, "DISASTER"])
    
    updated_jsonl_data.append([content, {"entities": entities}])

output_file = "../datasets/train/2xfiltered_tagged_gpe.jsonl"

with open(output_file, "w") as jsonl_file:
    for entry in updated_jsonl_data:
        json.dump(entry, jsonl_file)
        jsonl_file.write("\n")

Processing rows: 100%|██████████| 53816/53816 [00:23<00:00, 2285.41it/s]


In [17]:
def has_overlap(text, entities):
    for i, (start1, end1, _) in enumerate(entities):
        if start1 >= end1: 
            print("Invalid entity: start must be less than end")
            print(text)
            return True
        for j, (start2, end2, _) in enumerate(entities):
            if start2 >= end2: 
                print("Invalid entity: start must be less than end")
                print(text)
                return True
            if i != j and (start1 < end2 and start2 < end1):  
                return True
    return False

def filter_non_overlapping_lines(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
        for line in infile:
            data = json.loads(line)
            content = data[0]
            entities = data[1].get("entities", [])
            
            if not has_overlap(content, entities): 
                json.dump(data, outfile)
                outfile.write("\n")

In [18]:
input_file = "../datasets/train/2xfiltered_tagged_gpe.jsonl"
output_file = "../datasets/train/3xfiltered_tagged_gpe.jsonl"
filter_non_overlapping_lines(input_file, output_file)

In [19]:
def remove_empty_entities(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
        for line in infile:
            tweet = json.loads(line)

            if tweet[1]["entities"]:
                json.dump(tweet, outfile, ensure_ascii=False)
                outfile.write("\n")


In [20]:
input_file = "../datasets/train/3xfiltered_tagged_gpe.jsonl"  
output_file = "../datasets/train/4xfiltered_tagged_gpe.jsonl" 

remove_empty_entities(input_file, output_file)

In [21]:
input_file = "../datasets/train/1xfiltered_tagged_gpe.jsonl"
output_file = "../datasets/train/5xfiltered_tagged_gpe.jsonl"

def replace_gpe_with_hash(text, entities):
    text = list(text)  
    for start, end, entity_type in entities:
        if entity_type == "GPE":
            text[start:end] = "#" * (end - start)
    updated_text = "".join(text)  
    updated_text = re.sub(r"#+", "#", updated_text)
    return " ".join(updated_text.split())

with open(input_file, "r", encoding="utf-8") as infile, open(output_file, "w", encoding="utf-8") as outfile:
    for line in infile:
        data = json.loads(line.strip()) 
        text, annotations = data
        entities = annotations.get("entities", [])
        updated_text = replace_gpe_with_hash(text, entities)
        updated_data = [updated_text, {"entities": []}]
        json.dump(updated_data, outfile)
        outfile.write("\n")


In [22]:
jsonl_file = "../datasets/train/5xfiltered_tagged_gpe.jsonl"
csv_file = "../datasets/jpgpe.csv"
output_jsonl_file = "../datasets/train/output.jsonl"

locations = pd.read_csv(csv_file)["location"].fillna("").astype(str).tolist()

with open(jsonl_file, "r", encoding="utf-8") as f:
    jsonl_data = [json.loads(line.strip()) for line in f]

location_index = 0

for i in range(len(jsonl_data)):
    text, annotation = jsonl_data[i]
    entities = annotation["entities"]

    while "#" in text and location_index < len(locations):
        location = locations[location_index]
        hashtag_index = text.index("#")
        text = text[:hashtag_index] + location + text[hashtag_index + 1:]

        start = hashtag_index
        end = hashtag_index + len(location)
        entities.append([start, end, "GPE"])
        location_index += 1

    jsonl_data[i] = [text, {"entities": entities}]

with open(output_jsonl_file, "w", encoding="utf-8") as f:
    for entry in jsonl_data:
        json.dump(entry, f)
        f.write("\n")

print(f"Processed file saved as {output_jsonl_file}")


Processed file saved as ../datasets/train/output.jsonl


In [23]:
input_file = "../datasets/train/output.jsonl" 
jsonl_data = []

with open(input_file, "r", encoding="utf-8") as file:
    for line in file:
        jsonl_data.append(json.loads(line))

updated_jsonl_data = []

for entry in tqdm(jsonl_data, desc="Processing rows"):
    content = entry[0]
    entities = entry[1]["entities"]

    for keyword in disaster_keywords:
        if isinstance(keyword, str):
            lowercase_keyword = keyword.lower()
        else:
            lowercase_keyword = str(keyword).lower()
        
        pattern = r'\b' + re.escape(lowercase_keyword) + r'\b'
        matches = re.finditer(pattern, content.lower())
        for match in matches:
            start_idx = match.start()
            end_idx = match.end()
            if [start_idx, end_idx, "DISASTER"] not in entities:
                entities.append([start_idx, end_idx, "DISASTER"])
    
    updated_jsonl_data.append([content, {"entities": entities}])

output_file = "../datasets/train/output2.jsonl"

with open(output_file, "w") as jsonl_file:
    for entry in updated_jsonl_data:
        json.dump(entry, jsonl_file)
        jsonl_file.write("\n")

Processing rows: 100%|██████████| 53816/53816 [00:23<00:00, 2302.43it/s]


In [24]:
input_file = "../datasets/train/output2.jsonl"  
output_file = "../datasets/train/output3.jsonl"  

remove_empty_entities(input_file, output_file)

In [25]:
input_file = "../datasets/train/output3.jsonl"
output_file = "../datasets/train/output4.jsonl"
filter_non_overlapping_lines(input_file, output_file)

Invalid entity: start must be less than end
My heart goes out to the people of liu jia lu wu and mei tian. Prayers for all those who have been affected by this devastating earthquake minakuchicho sandaiji Turkiye  syriaearthquake
